**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 04 · 比较两个模型：配对、McNemar 与多重比较

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 用配对设计比较同一批题上的两个系统，并解释它为什么更有效
- 对二元结局使用精确 McNemar 检验
- 在多个维度同时比较时控制多重比较
- 区分统计显著和实际重要，写出可执行的决策规则

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 用模拟研究数据比较系统 A 与系统 B

In [ ]:
from course.data import simulate_counselor_study, DIMENSIONS
from course.stats import paired_compare, mcnemar_exact, p_adjust
study = simulate_counselor_study(seed=0)
h = study["human"]
# 三位咨询师的平均分,作为每条回复的分数
avg = h[h.dim != "gate"].groupby(["item", "system", "dim"]).score.mean().unstack("system")
avg.head()

In [ ]:
res = []
for d in DIMENSIONS:
    sub = avg.xs(d, level="dim")
    pc = paired_compare(sub["A"].values, sub["B"].values)
    res.append({"dim": d, "diff A-B": pc["diff"], "ci_low": pc["ci_low"], "ci_high": pc["ci_high"], "p": pc["p_paired_t"]})
res = pd.DataFrame(res); res

**练习 4.1**：用 Holm 方法校正上表的 5 个 p 值（`p_adjust(..., "holm")`)，有几个维度在校正后仍然 p < 0.05?

In [ ]:
ans_4_1 = int((p_adjust(res.p.values, "holm") < 0.05).sum())
check("练习 4.1 Holm 校正后显著的维度数", ans_4_1, int((p_adjust(res.p.values, "holm") < 0.05).sum()))

## 2. 配对与独立：区间宽度对比

In [ ]:
sub = avg.xs("D4", level="dim")
pc = paired_compare(sub["A"].values, sub["B"].values)
print("配对区间宽度:", round(pc["ci_high"] - pc["ci_low"], 3))
print("当作独立样本的区间宽度:", round(pc["unpaired_ci"][1] - pc["unpaired_ci"][0], 3))

**练习 4.2**：独立区间宽度是配对区间宽度的几倍？（保留两位小数）

In [ ]:
ans_4_2 = (pc["unpaired_ci"][1] - pc["unpaired_ci"][0]) / (pc["ci_high"] - pc["ci_low"])
check("练习 4.2 宽度之比", ans_4_2, (pc["unpaired_ci"][1] - pc["unpaired_ci"][0]) / (pc["ci_high"] - pc["ci_low"]), tol=0.05)

## 3. 安全门槛：McNemar 检验

用咨询师 c1 的安全门槛判断（1 = 通过）。

In [ ]:
gate = h[(h.dim == "gate") & (h.rater == "c1")].pivot_table(index="item", columns="system", values="score")
print("通过率:", gate.mean().round(3).to_dict())
print(mcnemar_exact(gate["A"], gate["B"]))

**练习 4.3**：只看风险题（`risk_flag == 1`）。重算两个系统的通过率和 McNemar p 值。填入 p 值，并思考：题数这么少时，这个检验能发现多大的差异？

In [ ]:
risky_items = study["replies"].query("risk_flag == 1")["item"].unique()
g_r = gate.loc[gate.index.isin(risky_items)]
ans_4_3 = mcnemar_exact(g_r["A"], g_r["B"])["p"]
check("练习 4.3 风险题 McNemar p", ans_4_3, mcnemar_exact(g_r["A"], g_r["B"])["p"], tol=1e-6)

## 4. 决策规则

**练习 4.4**：写一个函数 `decide(diff, ci_low, ci_high, mid)`，规则如下：
- 区间下限 > `mid`：返回 `"improved"`
- 区间上限 < `-mid`：返回 `"worse"`
- 区间完全落在 `(-mid, mid)` 内：返回 `"equivalent"`
- 其他：返回 `"inconclusive"`

In [ ]:
def decide(diff, ci_low, ci_high, mid):
    if ci_low > mid:
        return "improved"
    if ci_high < -mid:
        return "worse"
    if -mid < ci_low and ci_high < mid:
        return "equivalent"
    return "inconclusive"

tests = [((0.5, 0.3, 0.7, 0.2), "improved"), ((-0.5, -0.8, -0.3, 0.2), "worse"),
         ((0.0, -0.1, 0.1, 0.2), "equivalent"), ((0.1, -0.2, 0.4, 0.2), "inconclusive")]
check("练习 4.4 决策函数", [decide(*a) for a, _ in tests], [e for _, e in tests])

## 口述题（录音，每题 90 秒）

1. 训练团队报告新模型提升 2 分，你怎么判断？
   <details><summary>参考要点</summary>配对比较给出差值和区间；多次运行检查采样方差；多维度用 Holm 校正；确认裁判版本没变、测试集没被污染；最后对照事先定好的最小重要差异。</details>
2. 为什么二元通过率要用 McNemar 而不是卡方？
   <details><summary>参考要点</summary>两个系统做的是同一批题，结果配对。只有不一致的题提供差异信息。卡方把两组当独立样本，忽略了配对。</details>

## 研究延伸

CBT-Bench 以英文和一般成人场景为主。把类似的分级框架用于中文和老年人场景（例如老年人常见的认知扭曲和核心信念），并用配对设计比较模型，是一个与你的研究直接衔接的方向。

In [ ]:
summary()